# Voice Noise Analysis Round 2

这一版 notebook 对应当前第二轮素材，重点是：

- 使用更新后的 `10.83s` 音频
- 同时查看 `clean / hiss / noisy / filtered(3200) / filtered(3600)`
- 检查 waveform、FFT magnitude spectrum、spectrogram
- 观察 `3200 Hz` 和 `3600 Hz` 的差别

这份 notebook 不覆盖 `round1`，只是作为第二版分析入口。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from scipy import signal
from IPython.display import Audio, display

base = Path(r'/Users/xiejz959/Xiejz/College/Signal System & Probability/project/SSP_CP/Codes')
audio_dir = base / 'generated_audio'
chart_dir = Path(r'/Users/xiejz959/Xiejz/College/Signal System & Probability/project/SSP_CP/Charts/analysis_round1')

files = {
    'Clean Voice': audio_dir / 'clean_voice.wav',
    'Hiss Noise': audio_dir / 'hiss_noise.wav',
    'Noisy Voice': audio_dir / 'noisy_voice.wav',
    'Filtered Voice (3200 Hz)': audio_dir / 'filtered_voice_3200.wav',
    'Filtered Voice (3600 Hz)': audio_dir / 'filtered_voice_3600.wav',
}

signals = {}
for name, path in files.items():
    x, sr = sf.read(path)
    if x.ndim > 1:
        x = x.mean(axis=1)
    signals[name] = (x.astype(float), sr)
    print(f"{name}: sr={sr}, duration={len(x)/sr:.2f} s, samples={len(x)}")


In [ ]:
for name, (x, sr) in signals.items():
    print(name)
    display(Audio(x, rate=sr))


## 1. 全长 waveform

先看全长波形，确认新的音频长度已经超过 10 秒，同时观察 noisy 与 filtered 的整体差异。

In [ ]:
fig, axes = plt.subplots(len(signals), 1, figsize=(14, 11), sharex=True)
colors = {
    'Clean Voice': 'tab:blue',
    'Hiss Noise': 'tab:orange',
    'Noisy Voice': 'tab:red',
    'Filtered Voice (3200 Hz)': 'tab:green',
    'Filtered Voice (3600 Hz)': 'tab:purple',
}

for ax, (name, (x, sr)) in zip(axes, signals.items()):
    t = np.arange(len(x)) / sr
    ax.plot(t, x, color=colors.get(name, 'black'), linewidth=0.7)
    ax.set_title(name)
    ax.set_ylabel('Amp.')
    ax.grid(alpha=0.25)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Waveform Overview (Round 2)', fontsize=15)
plt.tight_layout()
plt.show()


## 2. Demo 片段 waveform

为了更接近上台展示，这里只看前 4 秒。这个视角更容易看出 `Noisy` 和 `Filtered` 在时域上的差别。

In [ ]:
demo_sec = 4.0
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
selected_names = ['Clean Voice', 'Noisy Voice', 'Filtered Voice (3200 Hz)']

for ax, name in zip(axes, selected_names):
    x, sr = signals[name]
    n = min(len(x), int(demo_sec * sr))
    t = np.arange(n) / sr
    ax.plot(t, x[:n], color=colors[name], linewidth=0.8)
    ax.set_title(name)
    ax.set_ylabel('Amp.')
    ax.grid(alpha=0.25)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Demo Window Waveforms (First 4 Seconds)', fontsize=15)
plt.tight_layout()
plt.show()


## 3. FFT magnitude spectrum

这一部分主要用来观察：

- clean voice 的主要能量集中在哪里
- hiss noise 是否在高频更明显
- noisy voice 的高频是否被抬高
- filtered 之后高频是否被压下去

In [ ]:
def magnitude_spectrum(x, fs):
    freqs = np.fft.rfftfreq(len(x), 1 / fs)
    mag = np.abs(np.fft.rfft(x))
    return freqs, mag

fig, axes = plt.subplots(len(signals), 1, figsize=(14, 12), sharex=True)
for ax, (name, (x, sr)) in zip(axes, signals.items()):
    f, mag = magnitude_spectrum(x, sr)
    ax.plot(f, mag, color=colors.get(name, 'black'), linewidth=0.8)
    ax.set_title(name)
    ax.set_ylabel('Magnitude')
    ax.grid(alpha=0.25)

axes[-1].set_xlabel('Frequency (Hz)')
axes[-1].set_xlim(0, sr / 2)
plt.suptitle('FFT Magnitude Spectrum (Round 2)', fontsize=15)
plt.tight_layout()
plt.show()


## 4. Spectrogram

这部分更适合看高频 hiss 的“持续背景感”。

建议重点比较：
- `Clean Voice`
- `Noisy Voice`
- `Filtered Voice (3200 Hz)`
- `Filtered Voice (3600 Hz)`

In [ ]:
def draw_spec(ax, x, fs, title):
    freqs, times, spec = signal.spectrogram(
        x,
        fs=fs,
        window='hann',
        nperseg=512,
        noverlap=384,
        scaling='spectrum',
        mode='magnitude',
    )
    spec_db = 20 * np.log10(spec + 1e-8)
    im = ax.pcolormesh(times, freqs, spec_db, shading='gouraud', cmap='magma')
    ax.set_title(title)
    ax.set_ylabel('Frequency (Hz)')
    ax.set_ylim(0, fs / 2)
    return im

spec_names = ['Clean Voice', 'Noisy Voice', 'Filtered Voice (3200 Hz)', 'Filtered Voice (3600 Hz)']
fig, axes = plt.subplots(len(spec_names), 1, figsize=(14.5, 12), sharex=True)
img = None
for ax, name in zip(axes, spec_names):
    x, sr = signals[name]
    img = draw_spec(ax, x, sr, name)

axes[-1].set_xlabel('Time (s)')
fig.subplots_adjust(right=0.88, hspace=0.28)
cax = fig.add_axes([0.90, 0.15, 0.025, 0.70])
fig.colorbar(img, cax=cax, format='%+2.0f dB')
plt.suptitle('Spectrogram Comparison (Round 2)', fontsize=15)
plt.show()


## 5. 3200 Hz vs 3600 Hz 对比

这里直接比较两种 cutoff 的结果，帮助决定课堂主 demo 用哪一版。

In [ ]:
clean, fs = signals['Clean Voice']
filtered_3200, _ = signals['Filtered Voice (3200 Hz)']
filtered_3600, _ = signals['Filtered Voice (3600 Hz)']


def snr_db(reference, estimate):
    error = estimate - reference
    return 10.0 * np.log10(np.sum(reference**2) / np.sum(error**2))

print('SNR with 3200 Hz:', round(snr_db(clean, filtered_3200), 3), 'dB')
print('SNR with 3600 Hz:', round(snr_db(clean, filtered_3600), 3), 'dB')
print('Correlation with clean (3200 Hz):', round(np.corrcoef(clean, filtered_3200)[0, 1], 5))
print('Correlation with clean (3600 Hz):', round(np.corrcoef(clean, filtered_3600)[0, 1], 5))

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, name in zip(axes, ['Filtered Voice (3200 Hz)', 'Filtered Voice (3600 Hz)']):
    x, sr = signals[name]
    n = min(len(x), int(4.0 * sr))
    t = np.arange(n) / sr
    ax.plot(t, x[:n], color=colors[name], linewidth=0.8)
    ax.set_title(name)
    ax.set_ylabel('Amp.')
    ax.grid(alpha=0.25)
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Filtered Result Comparison (First 4 Seconds)', fontsize=15)
plt.tight_layout()
plt.show()


## 6. 当前推荐结论

如果这版 notebook 跑出来和脚本结果一致，那么目前可以继续保留：

- 主 demo cutoff：`3200 Hz`
- `3600 Hz` 作为补充比较版本

原因不是 `3600 Hz` 完全不好，而是 `3200 Hz` 在课堂展示里更容易体现“噪声抑制与语音保留之间的平衡”。